# Confidence Distribution Analysis: Baseline vs DualCache

对比 **Baseline**（无 dualcache，每步做完整 forward）和 **DualCache**（KV-cache block-wise）两种生成模式下，
每个 step 中所有 masked token 的 **confidence score**（`softmax(logits)[predicted_token]`）的分布。

- 数据集: GSM8K test (100 samples, 5-shot)
- 参数: `gen_length=256, steps=256, block_length=32, threshold=0.9`
- 模型: `GSAI-ML/LLaDA-8B-Instruct`

## 1. 环境设置

In [ ]:
import os
import sys
import torch
import gc

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), 'llada'))
print(f'Working dir: {os.getcwd()}')

torch.cuda.empty_cache()
gc.collect()

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 2. 加载模型 & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from model.modeling_llada import LLaDAModelLM

MODEL_PATH = 'GSAI-ML/LLaDA-8B-Instruct'
MASK_ID = 126336

config = AutoConfig.from_pretrained(MODEL_PATH)
config.flash_attention = True

model = LLaDAModelLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True,
    torch_dtype=torch.bfloat16, config=config,
).eval().to('cuda')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f'Model loaded: {MODEL_PATH}')

## 3. 加载 GSM8K + 构造 5-shot Prompt

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset('gsm8k', 'main', split='test')
print(f'GSM8K test size: {len(gsm8k)}')

FEW_SHOT_EXAMPLES = """Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend $1.5 x 4 = $<<1.5*4=6>>6 for the soda.
Elle and Andrea will spend $6 + $10 = $<<6+10=16>>16 for the soda and chicken wings.
Elle, Andrea, and Joe together will spend $16 + $5 = $<<16+5=21>>21.
So, Mary will spend $22 - $21 = $<<22-21=1>>1 more than all of them combined.
#### 1

Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?
Answer: The grill burned 3 * 60 = <<3*60=180>>180 coals.
It takes 20 minutes to burn 15 coals, so the grill ran for 180 / 15 * 20 = <<180/15*20=240>>240 minutes.
#### 240

Question: A bear is preparing to hibernate for the winter and needs to gain 1000 pounds. At the end of summer, the bear feasts on berries and small woodland animals. During autumn, it devours acorns and salmon. It gained a fifth of the weight it needed from berries during summer, and during autumn, it gained twice that amount from acorns. Salmon made up half of the remaining weight it had needed to gain. How many pounds did it gain eating small animals?
Answer: The bear gained 1 / 5 * 1000 = <<1/5*1000=200>>200 pounds from berries.
It gained 2 * 200 = <<2*200=400>>400 pounds from acorns.
It still needed 1000 - 200 - 400 = <<1000-200-400=400>>400 pounds.
Thus, it gained 400 / 2 = <<400/2=200>>200 pounds from salmon.
Therefore, the bear gained 400 - 200 = <<400-200=200>>200 pounds from small animals.
#### 200

Question: Brendan can cut 8 yards of grass per day, he bought a lawnmower and it helped him to cut more yards by Fifty percent per day. How many yards will Brendan be able to cut after a week?
Answer: The additional yard Brendan can cut after buying the lawnmower is 8 x 0.50 = <<8*0.50=4>>4 yards.
So, the total yards he can cut with the lawnmower is 8 + 4 = <<8+4=12>>12.
Therefore, the total number of yards he can cut in a week is 12 x 7 = <<12*7=84>>84 yards.
#### 84"""


def build_prompt(question: str) -> torch.Tensor:
    """构造 5-shot instruct prompt 并 tokenize"""
    text = FEW_SHOT_EXAMPLES + f'\n\nQuestion: {question}\nAnswer:'
    messages = [{'role': 'user', 'content': text}]
    formatted = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False,
    )
    input_ids = tokenizer(formatted)['input_ids']
    return torch.tensor(input_ids, dtype=torch.long, device='cuda').unsqueeze(0)


LIMIT = 100
prompts = []
for i in range(LIMIT):
    prompts.append(build_prompt(gsm8k[i]['question']))

print(f'Built {len(prompts)} prompts, first prompt length: {prompts[0].shape[1]} tokens')

## 4. 工具函数（复用自 generate.py）

In [ ]:
import torch.nn.functional as F
import numpy as np

from generate import add_gumbel_noise, get_num_transfer_tokens, get_transfer_index

## 5. Baseline 生成（无 dualcache）+ Confidence 记录

基于原始 `generate()` 逻辑，额外在每个 step 收集所有 masked token 的 confidence。

In [ ]:
@torch.no_grad()
def generate_baseline_with_conf(
    model, prompt, steps=256, gen_length=256, block_length=32,
    temperature=0., remasking='low_confidence', mask_id=MASK_ID, threshold=0.9,
):
    """
    与 generate() 逻辑一致，但额外收集每个 step 中
    所有 masked token 的 confidence（softmax prob of argmax token）。
    """
    x = torch.full(
        (prompt.shape[0], prompt.shape[1] + gen_length),
        mask_id, dtype=torch.long, device=model.device,
    )
    x[:, :prompt.shape[1]] = prompt.clone()

    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    conf_records = []  # [{step, block, confidences, num_masked, num_transferred}]
    nfe = 0
    global_step = 0

    for nb in range(num_blocks):
        block_start = prompt.shape[1] + nb * block_length
        block_end = block_start + block_length

        block_mask_index = (x[:, block_start:block_end] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)

        i = 0
        while True:
            nfe += 1
            mask_index = (x == mask_id)
            logits = model(x).logits
            mask_index[:, block_end:] = False

            # ---- 收集 confidence ----
            x0_greedy = torch.argmax(logits, dim=-1)
            p = F.softmax(logits.to(torch.float64), dim=-1)
            x0_p = torch.gather(p, dim=-1, index=x0_greedy.unsqueeze(-1)).squeeze(-1)
            masked_conf = x0_p[mask_index].float().cpu().numpy()

            # ---- 正常 transfer ----
            quota = None if threshold is not None else num_transfer_tokens[:, i]
            x0, transfer_index = get_transfer_index(
                logits, temperature, remasking, mask_index, x, quota, threshold,
            )
            x[transfer_index] = x0[transfer_index]

            conf_records.append({
                'step': global_step,
                'block': nb,
                'confidences': masked_conf,
                'num_masked': int(mask_index.sum().item()),
                'num_transferred': int(transfer_index.sum().item()),
            })

            global_step += 1
            i += 1
            if (x[:, block_start:block_end] == mask_id).sum() == 0:
                break

    return x, nfe, conf_records

## 6. DualCache 生成 + Confidence 记录

基于 `generate_with_dual_cache()` 逻辑，额外收集 confidence。

DualCache 的核心区别：
- Step 0: 完整 forward（warm cache），在全局 mask 上做 transfer
- Step 1+: 仅对当前 block 做 forward（复用 KV-cache），在 block mask 上做 transfer

In [ ]:
@torch.no_grad()
def generate_dualcache_with_conf(
    model, prompt, steps=256, gen_length=256, block_length=32,
    temperature=0., remasking='low_confidence', mask_id=MASK_ID, threshold=0.9,
):
    """
    与 generate_with_dual_cache() 逻辑一致，但额外收集每个 step 的 confidence。
    """
    B = prompt.shape[0]
    Lp = int(prompt.shape[1])
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    x = torch.full((B, Lp + gen_length), mask_id, dtype=torch.long, device=model.device)
    x[:, :Lp] = prompt

    conf_records = []
    nfe = 0
    global_step = 0

    for nb in range(num_blocks):
        s = Lp + nb * block_length
        e = s + block_length

        block_mask_index = (x[:, s:e] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)

        # ---- Step 0: warm KV-cache on full sequence ----
        out_full = model(x, use_cache=True)
        past_key_values = out_full.past_key_values
        nfe += 1

        replace_position = torch.zeros_like(x, dtype=torch.bool)
        replace_position[:, s:e] = True

        global_mask_index = (x == mask_id)
        global_mask_index[:, e:] = False

        # Confidence on full logits (global mask)
        x0_greedy = torch.argmax(out_full.logits, dim=-1)
        p = F.softmax(out_full.logits.to(torch.float64), dim=-1)
        x0_p = torch.gather(p, dim=-1, index=x0_greedy.unsqueeze(-1)).squeeze(-1)
        masked_conf = x0_p[global_mask_index].float().cpu().numpy()

        quota0 = None if threshold is not None else num_transfer_tokens[:, 0]
        x0, transfer_index = get_transfer_index(
            out_full.logits, temperature, remasking, global_mask_index, x, quota0, threshold,
        )
        x = torch.where(transfer_index, x0, x)

        conf_records.append({
            'step': global_step,
            'block': nb,
            'step_type': 'warm',
            'confidences': masked_conf,
            'num_masked': int(global_mask_index.sum().item()),
            'num_transferred': int(transfer_index.sum().item()),
        })
        global_step += 1

        # ---- Step 1+: refine with KV-cache (block-only forward) ----
        for i in range(1, steps_per_block):
            if (x[:, s:e] == mask_id).sum() == 0:
                break

            logits_blk = model(
                x[:, s:e], past_key_values=past_key_values,
                use_cache=True, replace_position=replace_position,
            ).logits

            mask_blk = (x[:, s:e] == mask_id)

            # Confidence on block logits
            x0_greedy_blk = torch.argmax(logits_blk, dim=-1)
            p_blk = F.softmax(logits_blk.to(torch.float64), dim=-1)
            x0_p_blk = torch.gather(p_blk, dim=-1, index=x0_greedy_blk.unsqueeze(-1)).squeeze(-1)
            masked_conf_blk = x0_p_blk[mask_blk].float().cpu().numpy()

            quota_i = None if threshold is not None else num_transfer_tokens[:, i]
            x0_blk, transfer_idx_blk = get_transfer_index(
                logits_blk, temperature, remasking, mask_blk, x[:, s:e], quota_i, threshold,
            )

            blk_old = x[:, s:e]
            blk_new = torch.where(transfer_idx_blk, x0_blk, blk_old)
            x = torch.cat([x[:, :s], blk_new, x[:, e:]], dim=1)

            nfe += 1
            conf_records.append({
                'step': global_step,
                'block': nb,
                'step_type': 'refine',
                'confidences': masked_conf_blk,
                'num_masked': int(mask_blk.sum().item()),
                'num_transferred': int(transfer_idx_blk.sum().item()),
            })
            global_step += 1

    return x, nfe, conf_records

## 7. 运行实验：收集 Confidence 分布

依次对 100 个 GSM8K 样本运行 Baseline 和 DualCache，收集所有 step 的 confidence。

先跑 Baseline，清空 cache 后再跑 DualCache（同一个模型，顺序跑即可）。

In [ ]:
import time
import json
from tqdm.auto import tqdm

GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
THRESHOLD = 0.9

gen_kwargs = dict(
    steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
    temperature=0., remasking='low_confidence', mask_id=MASK_ID, threshold=THRESHOLD,
)

# ---------- Baseline ----------
print('=' * 60)
print('Running Baseline (no dualcache) ...')
print('=' * 60)

baseline_records = []   # list of list-of-step-dicts, one per sample
baseline_nfes = []
t0 = time.time()

for idx in tqdm(range(LIMIT), desc='Baseline'):
    _, nfe, conf_rec = generate_baseline_with_conf(model, prompts[idx], **gen_kwargs)
    baseline_records.append(conf_rec)
    baseline_nfes.append(nfe)

baseline_time = time.time() - t0
print(f'Baseline done: {baseline_time:.1f}s, avg NFE={np.mean(baseline_nfes):.1f}')

torch.cuda.empty_cache()
gc.collect()

# ---------- DualCache ----------
print('\n' + '=' * 60)
print('Running DualCache ...')
print('=' * 60)

dualcache_records = []
dualcache_nfes = []
t0 = time.time()

for idx in tqdm(range(LIMIT), desc='DualCache'):
    _, nfe, conf_rec = generate_dualcache_with_conf(model, prompts[idx], **gen_kwargs)
    dualcache_records.append(conf_rec)
    dualcache_nfes.append(nfe)

dualcache_time = time.time() - t0
print(f'DualCache done: {dualcache_time:.1f}s, avg NFE={np.mean(dualcache_nfes):.1f}')

torch.cuda.empty_cache()

## 8. 保存原始数据（可选，方便后续离线分析）

In [ ]:
import pickle, os

save_dir = 'confidence_analysis'
os.makedirs(save_dir, exist_ok=True)

def records_to_serializable(records):
    """numpy arrays → lists for JSON/pickle compatibility"""
    out = []
    for sample_recs in records:
        sample_out = []
        for r in sample_recs:
            d = dict(r)
            d['confidences'] = d['confidences'].tolist() if hasattr(d['confidences'], 'tolist') else d['confidences']
            sample_out.append(d)
        out.append(sample_out)
    return out

with open(os.path.join(save_dir, 'baseline_records.pkl'), 'wb') as f:
    pickle.dump(records_to_serializable(baseline_records), f)
with open(os.path.join(save_dir, 'dualcache_records.pkl'), 'wb') as f:
    pickle.dump(records_to_serializable(dualcache_records), f)

print(f'Saved to {save_dir}/')

## 9. 数据聚合

In [ ]:
def flatten_confidences(records):
    """将所有 sample 的所有 step 的 confidence 展平为一个大数组"""
    all_conf = []
    for sample_recs in records:
        for r in sample_recs:
            vals = r['confidences']
            if isinstance(vals, np.ndarray):
                all_conf.append(vals)
            else:
                all_conf.append(np.array(vals, dtype=np.float32))
    return np.concatenate(all_conf) if all_conf else np.array([])


def per_step_stats(records):
    """
    计算每个 intra-block step index 的平均 confidence。
    返回 dict: step_index -> {mean, median, std, count}
    """
    from collections import defaultdict
    step_data = defaultdict(list)
    for sample_recs in records:
        local_step = 0
        prev_block = -1
        for r in sample_recs:
            if r['block'] != prev_block:
                local_step = 0
                prev_block = r['block']
            vals = r['confidences']
            if isinstance(vals, list):
                vals = np.array(vals, dtype=np.float32)
            if len(vals) > 0:
                step_data[local_step].append(vals)
            local_step += 1

    stats = {}
    for si in sorted(step_data.keys()):
        combined = np.concatenate(step_data[si])
        stats[si] = {
            'mean': float(np.mean(combined)),
            'median': float(np.median(combined)),
            'std': float(np.std(combined)),
            'count': len(combined),
        }
    return stats


baseline_all = flatten_confidences(baseline_records)
dualcache_all = flatten_confidences(dualcache_records)

print(f'Baseline  总 confidence 数据点: {len(baseline_all):,}')
print(f'DualCache 总 confidence 数据点: {len(dualcache_all):,}')
print()
print(f'Baseline  mean={baseline_all.mean():.4f}, median={np.median(baseline_all):.4f}, std={baseline_all.std():.4f}')
print(f'DualCache mean={dualcache_all.mean():.4f}, median={np.median(dualcache_all):.4f}, std={dualcache_all.std():.4f}')
print()
print(f'Baseline  >= 0.9 比例: {(baseline_all >= THRESHOLD).mean():.2%}')
print(f'DualCache >= 0.9 比例: {(dualcache_all >= THRESHOLD).mean():.2%}')

## 10. 可视化

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ---- (a) 总体 Histogram ----
ax = axes[0, 0]
ax.hist(baseline_all, bins=100, range=(0, 1), alpha=0.55, label='Baseline', density=True, color='steelblue')
ax.hist(dualcache_all, bins=100, range=(0, 1), alpha=0.55, label='DualCache', density=True, color='coral')
ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'threshold={THRESHOLD}')
ax.set_xlabel('Confidence')
ax.set_ylabel('Density')
ax.set_title('(a) Overall Confidence Distribution')
ax.legend()

# ---- (b) CDF ----
ax = axes[0, 1]
for data, label, color in [
    (baseline_all, 'Baseline', 'steelblue'),
    (dualcache_all, 'DualCache', 'coral'),
]:
    sorted_d = np.sort(data)
    cdf = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
    # Subsample for plotting efficiency
    step = max(1, len(sorted_d) // 5000)
    ax.plot(sorted_d[::step], cdf[::step], label=label, color=color, linewidth=1.5)
ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'threshold={THRESHOLD}')
ax.set_xlabel('Confidence')
ax.set_ylabel('CDF')
ax.set_title('(b) Cumulative Distribution')
ax.legend(loc='lower right')

# ---- (c) Mean Confidence per Intra-Block Step ----
ax = axes[1, 0]
for records, label, color in [
    (baseline_records, 'Baseline', 'steelblue'),
    (dualcache_records, 'DualCache', 'coral'),
]:
    stats = per_step_stats(records)
    steps_x = sorted(stats.keys())
    means = [stats[s]['mean'] for s in steps_x]
    stds = [stats[s]['std'] for s in steps_x]
    ax.plot(steps_x, means, label=label, color=color, linewidth=1.5)
    ax.fill_between(
        steps_x,
        [m - sd for m, sd in zip(means, stds)],
        [m + sd for m, sd in zip(means, stds)],
        alpha=0.15, color=color,
    )
ax.axhline(THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'threshold={THRESHOLD}')
ax.set_xlabel('Intra-Block Step Index')
ax.set_ylabel('Mean Confidence')
ax.set_title('(c) Mean Confidence per Intra-Block Step (±1σ)')
ax.legend()

# ---- (d) Per-Step Unmask Ratio ----
ax = axes[1, 1]
for records, label, color in [
    (baseline_records, 'Baseline', 'steelblue'),
    (dualcache_records, 'DualCache', 'coral'),
]:
    from collections import defaultdict
    step_ratios = defaultdict(list)
    for sample_recs in records:
        local_step = 0
        prev_block = -1
        for r in sample_recs:
            if r['block'] != prev_block:
                local_step = 0
                prev_block = r['block']
            if r['num_masked'] > 0:
                step_ratios[local_step].append(r['num_transferred'] / r['num_masked'])
            local_step += 1
    steps_x = sorted(step_ratios.keys())
    means = [np.mean(step_ratios[s]) for s in steps_x]
    ax.plot(steps_x, means, label=label, color=color, linewidth=1.5)
ax.set_xlabel('Intra-Block Step Index')
ax.set_ylabel('Transfer Ratio (transferred / masked)')
ax.set_title('(d) Per-Step Unmask Ratio')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'confidence_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_dir}/confidence_distribution.png')

## 11. 细分视角：高/低 Confidence 区间

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Zoomed-in: high confidence region [0.7, 1.0]
ax = axes[0]
ax.hist(baseline_all[baseline_all >= 0.7], bins=60, range=(0.7, 1.0), alpha=0.55,
        label='Baseline', density=True, color='steelblue')
ax.hist(dualcache_all[dualcache_all >= 0.7], bins=60, range=(0.7, 1.0), alpha=0.55,
        label='DualCache', density=True, color='coral')
ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'threshold={THRESHOLD}')
ax.set_xlabel('Confidence')
ax.set_ylabel('Density')
ax.set_title('Zoom: High Confidence Region [0.7, 1.0]')
ax.legend()

# (b) Zoomed-in: low confidence region [0, 0.3]
ax = axes[1]
ax.hist(baseline_all[baseline_all <= 0.3], bins=60, range=(0, 0.3), alpha=0.55,
        label='Baseline', density=True, color='steelblue')
ax.hist(dualcache_all[dualcache_all <= 0.3], bins=60, range=(0, 0.3), alpha=0.55,
        label='DualCache', density=True, color='coral')
ax.set_xlabel('Confidence')
ax.set_ylabel('Density')
ax.set_title('Zoom: Low Confidence Region [0, 0.3]')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'confidence_zoomed.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. NFE & Step 统计

In [ ]:
def count_steps_per_sample(records):
    return [len(sample_recs) for sample_recs in records]

baseline_steps = count_steps_per_sample(baseline_records)
dualcache_steps = count_steps_per_sample(dualcache_records)

print('Steps per sample (total across all blocks):')
print(f'  Baseline  — mean: {np.mean(baseline_steps):.1f}, min: {np.min(baseline_steps)}, max: {np.max(baseline_steps)}')
print(f'  DualCache — mean: {np.mean(dualcache_steps):.1f}, min: {np.min(dualcache_steps)}, max: {np.max(dualcache_steps)}')
print()
print('NFE per sample:')
print(f'  Baseline  — mean: {np.mean(baseline_nfes):.1f}, min: {np.min(baseline_nfes)}, max: {np.max(baseline_nfes)}')
print(f'  DualCache — mean: {np.mean(dualcache_nfes):.1f}, min: {np.min(dualcache_nfes)}, max: {np.max(dualcache_nfes)}')

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(baseline_nfes, bins=30, alpha=0.55, label='Baseline', color='steelblue')
ax.hist(dualcache_nfes, bins=30, alpha=0.55, label='DualCache', color='coral')
ax.set_xlabel('NFE per Sample')
ax.set_ylabel('Count')
ax.set_title('NFE Distribution (100 samples)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'nfe_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()